# recipe-dataclass — ex1: define Recipe and construct it for log_forward

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `recipe-dataclass`. Running the final beacon cell reports progress against the `Backprop: Recipe dataclass` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Recipe dataclass` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`recipe-dataclass`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "recipe-dataclass"
DD_SUBTOPIC = "Backprop: Recipe dataclass"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Recipe dataclass — quick refresher

Each non-leaf `Tensor` carries a `Recipe` that records exactly enough to replay the forward call in reverse. It's a 4-tuple:

```python
@dataclass(frozen=True)
class Recipe:
    func: Callable        # the forward fn (e.g. torch.log, torch.multiply)
    args: tuple           # raw positional args at call time (numbers / arrays)
    kwargs: dict          # raw keyword args at call time
    parents: dict[int, Tensor]  # argnum -> the parent Tensor (filtered)
```

`func` is the lookup key into `BACK_FUNCS`. `args` and `kwargs` are passed into the back fn so it can compute the local Jacobian. `parents` is the edge list of the computational graph — reverse traversal walks parents to find what to differentiate next.

**Always 4 fields, always in that order** — every wrap in the codebase constructs them the same way, so the reverse pass can read them generically.

### Exercise 1 — define Recipe and construct it for log_forward

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the Recipe-construction pattern by defining the 4-field dataclass and attaching a correctly-populated Recipe to the output of a single-arg forward op (log).
> Keywords: recipe, dataclass, log-forward, parents, func
> ```

**KCs targeted:** `recipe-dataclass`, `box-array-to-tensor-with-recipe`

Implement BOTH parts:

**1. The `Recipe` dataclass.** Define a `@dataclass` with EXACTLY these four fields, in this order:
   - `func: Callable` — the forward fn (e.g. `torch.log`).
   - `args: tuple` — the raw (unboxed) positional args at call time.
   - `kwargs: dict` — the raw keyword args at call time.
   - `parents: dict` — `{argnum: MiniTensor}` for each Tensor input.

**2. `log_forward(x)`** — single-arg autograd-aware log:
   - Accepts a `MiniTensor` whose `.array` is a `torch.Tensor`.
   - Computes `out_arr = torch.log(x.array)`.
   - Returns a new `MiniTensor(out_arr)` with `out.recipe` set to a `Recipe` carrying:
     - `func = torch.log`
     - `args = (x.array,)`  (a 1-tuple of the raw input)
     - `kwargs = {}`
     - `parents = {0: x}`  (arg 0 is the input MiniTensor)

**Why all four fields, always.** The reverse pass treats Recipe generically: it reads `recipe.func` to find the back fn, `recipe.args` and `recipe.kwargs` to replay the original call, and `recipe.parents` to find what to differentiate next. Drop any field and the dispatcher breaks.

Use plain `torch.Tensor` for `.array`; no autograd.

In [ ]:
@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict


def log_forward(x: MiniTensor) -> MiniTensor:
    out_arr = t.log(x.array)
    out = MiniTensor(out_arr)
    out.recipe = Recipe(
        func=t.log,
        args=(x.array,),     # raw, unboxed; 1-tuple
        kwargs={},            # log takes no kwargs in this drill
        parents={0: x},      # arg-0 is a Tensor → record it
    )
    return out


<details><summary>Solution</summary>

```python
@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict


def log_forward(x: MiniTensor) -> MiniTensor:
    out_arr = t.log(x.array)
    out = MiniTensor(out_arr)
    out.recipe = Recipe(
        func=t.log,
        args=(x.array,),     # raw, unboxed; 1-tuple
        kwargs={},            # log takes no kwargs in this drill
        parents={0: x},      # arg-0 is a Tensor → record it
    )
    return out
```

**Why a dataclass and not just a tuple.** A 4-tuple would work, but `@dataclass` gives you named attribute access (`recipe.func`, `recipe.parents`), repr-for-debug, and structural equality for free. The reverse pass would otherwise be a forest of `recipe[0]`, `recipe[1]` — opaque and bug-prone.

**`parents` is the edge list of the compute graph.** Every non-leaf Tensor's `recipe.parents` tells the reverse pass which Tensors fed into it. Topological sort follows these edges; the dispatcher then iterates `for argnum, parent in recipe.parents.items()` and calls the matching back fn at that argnum.

**Identity, not copy.** Storing `x.array` (the same tensor object) rather than `x.array.clone()` matters because (a) it's free, (b) elementwise back fns can read the cached input without allocating, and (c) some back fns even mutate in place during accumulation.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()